# Trainer demo launcher

Run the setup once. Choose a demo, select Prepare demo to inspect its starting evidence, then Run demo. Each preparation creates an isolated workspace. The 38 standalone notebooks remain available in the index.

Core demos provide the main route; optional demos add depth. Prepared mechanisms make no model call. Live responses may vary or fail. Made up, not from work: use the fictional records.


In [ ]:
#@title Open the course workspace
import os, sys, subprocess, json, urllib.request, importlib.util
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai==2.41.1', 'ipywidgets>=8,<9'], check=True)
BASE_URL = 'https://raw.githubusercontent.com/FeikoWielsma/Using-AI-Agents/main/'
def course_file(name):
    local_root = os.environ.get('COURSE_LOCAL_ROOT')
    if local_root:
        return Path(local_root) / name
    path = Path('course-assets') / name
    path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(BASE_URL + name, path)
    return path
KEY = os.environ.get('COURSE_API_KEY')
if not KEY:
    try:
        from google.colab import userdata
        KEY = userdata.get('COURSE_API_KEY')
    except Exception:
        KEY = None
from openai import OpenAI
MODEL = 'qwen3.8-max'
EXTRA = {'enable_thinking': False}
client = OpenAI(api_key=KEY or 'course-key-not-configured',
    base_url='https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1',
    timeout=35.0, max_retries=0)
spec=importlib.util.spec_from_file_location('course_workbench',course_file('notebooks/course_workbench.py'))
workbench=importlib.util.module_from_spec(spec)
spec.loader.exec_module(workbench)
print('Course interface ready.' if KEY else 'Add COURSE_API_KEY in Colab Secrets, enable notebook access, and rerun this setup. Source browsing remains available.')


In [ ]:
#@title Open the demonstration controls
import ipywidgets as widgets
from IPython.display import display, clear_output
spec=importlib.util.spec_from_file_location('demo_runtime',course_file('notebooks/demo_runtime.py'))
runtime=importlib.util.module_from_spec(spec);spec.loader.exec_module(runtime)
catalog=json.loads(course_file('data/demo-catalog.json').read_text(encoding='utf-8'))
packs=json.loads(course_file('data/case-packs.json').read_text(encoding='utf-8'))
traces=json.loads(course_file('data/agent-traces.json').read_text(encoding='utf-8'))
worked=course_file('data/worked-launch-workspace.zip').read_bytes()
choice=widgets.Dropdown(options=[(f"{d['id']:02} | M{d['module']} | {d['title']} | "+('Core' if d['core'] else 'Optional'),i) for i,d in enumerate(catalog)],layout=widgets.Layout(width='95%'))
prepare=widgets.Button(description='Prepare demo')
run=widgets.Button(description='Run demo',disabled=True)
output=widgets.Output()
state={'demo':None}
def reset(change):
    run.disabled=True
    state['demo']=None
def prepare_demo(_):
    with output:
        clear_output()
        d=catalog[choice.value]
        state['demo']=runtime.Demo(d,workbench,packs,traces,worked,client,MODEL,EXTRA)
        state['demo'].inspect()
        print('Observe:',d['observe'])
    run.disabled=False
def run_demo(_):
    choice.disabled=True;prepare.disabled=True;run.disabled=True
    try:
        with output:state['demo'].run()
    finally:
        choice.disabled=False;prepare.disabled=False
choice.observe(reset,names='value');prepare.on_click(prepare_demo);run.on_click(run_demo)
display(widgets.VBox([choice,widgets.HBox([prepare,run]),output]))
